In [1]:
import os

# Set the base directory for your cache
WORKSPACE_CACHE = "/workspace/cache"

# Ensure the directory exists
os.makedirs(WORKSPACE_CACHE, exist_ok=True)

# Point common libraries to this directory
os.environ["HF_HOME"] = os.path.join(WORKSPACE_CACHE, "huggingface")
os.environ["PIP_CACHE_DIR"] = os.path.join(WORKSPACE_CACHE, "pip")
os.environ["TORCH_HOME"] = os.path.join(WORKSPACE_CACHE, "torch")

In [2]:
import sys
!{sys.executable} -m pip install transformer_lens
!{sys.executable} -m pip install nltk
!{sys.executable} -m pip install hf_transfer

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.3/945.3 kB 21.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 10.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 15.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 78.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 70.5 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 76.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 100.1 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.6/806.6 kB 18.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.6/530.6 MB 93.6 MB/s  0:00:04m0:

In [3]:
import numpy as np
import torch
from torch.utils.data import DataLoader, IterableDataset, Dataset
import os
from transformer_lens import HookedTransformer
import nltk
import random
from tqdm import tqdm
from transformers import AutoModelForCausalLM
import json

/workspace/sae-binding/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ── CONFIG — only change these three lines to switch models ──────────────
TL_MODEL_NAME = "gemma-2-2b"         # TransformerLens model name
HF_MODEL_ID   = "google/gemma-2-2b"  # HuggingFace model ID for fine-tuning
OUTPUT_DIR    = "./gemma2_ft_toy"     # where the FT checkpoint is saved
# For Gemma 3-1b:  TL_MODEL_NAME="gemma-3-1b-pt"  HF_MODEL_ID="google/gemma-3-1b-pt"  OUTPUT_DIR="./gemma3_1b_ft_toy"
# For Gemma 3-4b:  TL_MODEL_NAME="gemma-3-4b-pt"  HF_MODEL_ID="google/gemma-3-4b-pt"  OUTPUT_DIR="./gemma3_4b_ft_toy"
# ─────────────────────────────────────────────────────────────────────────

In [15]:
os.environ["HF_TOKEN"] = input()
model = HookedTransformer.from_pretrained(TL_MODEL_NAME)

Loading weights: 100%|██████████| 288/288 [00:00<00:00, 4404.82it/s]


Loaded pretrained model gemma-2-2b into HookedTransformer


In [16]:
def read_relations_jsonl(path: str):
    """
    Read a JSONL file where each line has {"input": ..., "label": ...}.
    Returns a list of dicts.
    """
    records: List[Dict[str, str]] = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # skip blank lines
                records.append(json.loads(line))
    return records
    
text_examples = read_relations_jsonl("gemma_toy_dataset_train.jsonl")

In [17]:
def get_subset(text_examples, start_idx, end_idx):
    subset = []
    labels = []
    for rec in text_examples[start_idx: end_idx]:
        subset.append(model.to_tokens(rec["input"], prepend_bos=False).squeeze())
        labels.append(model.to_tokens(rec["label"], prepend_bos=False).squeeze().item())
    subset = torch.stack(subset, dim=0)
    return subset, labels

dataset, labels = get_subset(text_examples, 0, 8000)

In [18]:
def get_text_trainset(dataset, labels):
    train_dataset_text = []
    for idx in range(dataset.shape[0]):
        ex = dataset[idx]
        label = labels[idx]
        ex_text = model.to_string(ex)
        label_text = model.to_string(label)
        train_dataset_text.append({"prompt": ex_text, "label":label_text})
    return train_dataset_text

train_dataset_text = get_text_trainset(dataset, labels)

In [19]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch

model_id = HF_MODEL_ID
tok = model.tokenizer
CUE = ""   # or "" if you didn’t use a cue
raw = [ex for ex in train_dataset_text]

class FixedDataset(Dataset):
    def __init__(self, examples):
        self.recs = []
        for ex in examples:
            prompt_ids = tok(ex["prompt"], add_special_tokens=False)["input_ids"]
            label_ids  = tok(ex["label"], add_special_tokens=False)["input_ids"]  # leading space often helps

            # Build input_ids and labels (loss only on the label)
            input_ids = prompt_ids + label_ids
            labels    = [-100] * len(prompt_ids) + label_ids

            # Hard guarantee: all examples must have identical lengths
            self.recs.append({
                "input_ids": input_ids,
                "labels": labels,
            })

        # Sanity: assert fixed length
        lens_inp = {len(r["input_ids"]) for r in self.recs}
        lens_lab = {len(r["labels"]) for r in self.recs}
        assert len(lens_inp) == 1 and lens_inp == lens_lab, f"Lengths vary: {lens_inp=} {lens_lab=}"
        self.seq_len = next(iter(lens_inp))

    def __len__(self): return len(self.recs)
    def __getitem__(self, i): return self.recs[i]

train_ds = FixedDataset(raw[:-max(1, len(raw)//10)] or raw)
val_ds   = FixedDataset(raw[-max(1, len(raw)//10):] or raw[:min(100, len(raw))])


In [20]:
# ---- Collator that DOES NOT pad; just stacks (since lengths are identical) ----
def no_pad_collator(batch):
    input_ids = torch.tensor([ex["input_ids"] for ex in batch], dtype=torch.long)
    labels    = torch.tensor([ex["labels"]    for ex in batch], dtype=torch.long)
    # Optionally build attention_mask of ones
    attention_mask = torch.ones_like(input_ids)
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}




In [21]:
import gc
torch.cuda.empty_cache()
gc.collect()


2007

In [ ]:
# ---- Model & Trainer ----
auto_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    learning_rate=2e-5,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    #evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=40,
    report_to="none",
)

trainer = Trainer(
    model=auto_model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tok,
    data_collator=no_pad_collator,  # <- no padding added
)

# Quick shape sanity check
b = no_pad_collator([train_ds[0], train_ds[1]])
assert b["input_ids"].shape == b["labels"].shape  # [B, T]

trainer.train()

Loading weights: 100%|██████████| 288/288 [00:18<00:00, 15.67it/s]
/workspace/sae-binding/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
